In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

In [6]:
rng = np.random.default_rng(42)
n = 20_000

price = rng.gamma(4, 25, n)                                # hurts clicks
rating = np.clip(rng.normal(4.0, 0.6, n), 1, 5)            # helps clicks
delivery_days = rng.integers(1, 10, n).astype(float)       # mildly hurts
noise = rng.normal(0, 1, n)                                # irrelevant
brand = rng.choice(["acme", "globex", "initech", "umbrella"], n, p=[.4, .3, .2, .1])
device = rng.choice(["mobile", "desktop", "tablet"], n, p=[.6, .3, .1])  # irrelevant

brand_eff = {"acme": 0.3, "globex": 0.0, "initech": -0.3, "umbrella": 0.6}
logit = (-2.2
            - 0.012 * (price - 100)
            + 0.9 * (rating - 4.0)
            - 0.10 * (delivery_days - 5)
            + pd.Series(brand).map(brand_eff).to_numpy())
click = rng.binomial(1, 1 / (1 + np.exp(-logit)))

df = pd.DataFrame({
    "f_price": price, "f_rating": rating, "f_delivery_days": delivery_days,
    "f_noise": noise, "brand": brand, "device": device, "click": click,
})
print(df.shape)
df

(20000, 7)


,f_price,f_rating,f_delivery_days,f_noise,brand,device,click
0,107.041339,3.782051,5.0,1.740002,umbrella,mobile,0
1,132.489290,4.270216,1.0,0.436012,globex,mobile,0
2,97.923763,3.510630,1.0,-1.772765,globex,desktop,0
3,90.864723,5.000000,5.0,-1.532271,globex,mobile,0
4,140.537996,4.116207,7.0,-0.217699,acme,mobile,0
...,...,...,...,...,...,...,...
19995,110.143306,4.554287,4.0,0.552465,umbrella,mobile,0
19996,98.339055,3.884632,7.0,-1.821506,initech,mobile,0
19997,169.570182,2.858935,8.0,-0.806374,globex,mobile,0
19998,119.562165,3.950992,7.0,-1.236612,initech,desktop,1


In [8]:
click_col = 'click'

In [9]:
df[click_col] = pd.to_numeric(df[click_col]).astype(int)
if not set(df[click_col].unique()) <= {0, 1}:
    raise ValueError(f"'{click_col}' must contain only 0/1 values")

feature_cols = [c for c in df.columns if c != click_col]

categorical_cols = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(df[c])
    or pd.api.types.is_bool_dtype(df[c])
]

numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print(f'feature_cols: {feature_cols}')
print(f'categorical_cols; {categorical_cols}')
print(f'numeric_cols: {numeric_cols}')

feature_cols: ['f_price', 'f_rating', 'f_delivery_days', 'f_noise', 'brand', 'device']
categorical_cols; ['brand', 'device']
numeric_cols: ['f_price', 'f_rating', 'f_delivery_days', 'f_noise']


### Numeric columns

In [ ]:
def _benjamini_hochberg(pvals: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adj = p[order] * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]  # enforce monotonicity
    out = np.empty(n)
    out[order] = np.clip(adj, 0, 1)
    return out


def _cohens_d(x1: np.ndarray, x0: np.ndarray) -> float:
    """Standardized mean difference (pooled SD)."""
    n1, n0 = len(x1), len(x0)
    s1, s0 = x1.std(ddof=1), x0.std(ddof=1)
    pooled = np.sqrt(((n1 - 1) * s1**2 + (n0 - 1) * s0**2) / (n1 + n0 - 2))
    return 0.0 if pooled == 0 else (x1.mean() - x0.mean()) / pooled

In [ ]:
num_rows = []
for f in numeric_cols:
    sub = df[[f, click_col]].dropna()
    x1 = sub.loc[sub[click_col] == 1, f].astype(float).to_numpy()
    x0 = sub.loc[sub[click_col] == 0, f].astype(float).to_numpy()
    if len(x1) < 2 or len(x0) < 2 or sub[f].nunique() < 2:
        continue  # constant feature or missing class -> nothing to test

    # p_ttest_welch:p_t - p-value H₀: x0.mean() = x1.mean()
    # If the p-value is large, the data are consistent with H₀; we fail to reject H₀, but we cannot conclude that H₀ is true.
    # If the p-value is small, data as extreme as those observed would be very unlikely if H₀ were true; 
    #   therefore it is reasonable to reject H₀. 
    _, p_t = stats.ttest_ind(x1, x0, equal_var=False) 

    # if I grab one random clicked row and one random non-clicked row, does one of them tend to have the larger feature value?
    # θ=P(X1​>X0​)+0.5​*P(X1​=X0​) - the probability that a randomly drawn clicked value beats a randomly drawn non-clicked value
    # U1​=∑∑[1(x1i​>x0j​)+0.5*1(x1i​=x0j​)] θ_hat=U1/(n1*n2) n1=len(x1), n0=len(x0)
    # auc = θ_hat
    # p_u - p_value H₀: θ=0.5
    u_stat, p_u = stats.mannwhitneyu(x1, x0, alternative="two-sided")

    # Pearson's ordinary correlation coefficient applied to the special case where one of the two variables is binary (click ∈ {0,1})
    r_pb, _ = stats.pointbiserialr(sub[click_col].to_numpy(), sub[f].astype(float).to_numpy())

    # the AUC: U / (n1*n0) = P(value | click=1  >  value | click=0)
    auc = u_stat / (len(x1) * len(x0))
    
    diff = x1.mean() - x0.mean()

    num_rows.append({
        "feature": f,
        "n_used": len(sub),
        "mean_click1": x1.mean(),
        "mean_click0": x0.mean(),
        "mean_diff": diff,
        "direction": "higher when clicked" if diff > 0 else "lower when clicked",
        "cohens_d": _cohens_d(x1, x0),
        "auc_single_feature": auc,
        "pointbiserial_r": r_pb,
        "p_ttest_welch": p_t,
        "p_mannwhitney": p_u,
    })

num_res = pd.DataFrame(num_rows)
if not num_res.empty:
    num_res["p_adj_BH"] = _benjamini_hochberg(num_res["p_mannwhitney"])
    num_res = num_res.sort_values(
        "cohens_d", key=lambda s: s.abs(), ascending=False
    ).reset_index(drop=True)

In [13]:
num_res

,feature,n_used,mean_click1,mean_click0,mean_diff,direction,cohens_d,auc_single_feature,pointbiserial_r,p_ttest_welch,p_mannwhitney,p_adj_BH
0,f_price,20000,80.011468,103.674261,-23.662793,lower when clicked,-0.474614,0.360882,-0.160467,3.083202e-157,1.598862e-120,6.395450e-120
1,f_rating,20000,4.194492,3.947446,0.247046,higher when clicked,0.435503,0.623446,0.147544,8.130805e-103,2.535384e-95,5.070769e-95
2,f_delivery_days,20000,4.478453,5.095979,-0.617526,lower when clicked,-0.240519,0.431265,-0.082109,5.378980e-31,3.869233e-31,5.158977e-31
3,f_noise,20000,-0.001288,-0.009348,0.008060,higher when clicked,0.008017,0.500382,0.002746,6.984021e-01,9.488571e-01,9.488571e-01


### Categorical columns

While a Chi-Square test tells you if two categorical variables are related (statistical significance), Cramér's V tells you how strongly they are related (practical significance or "effect size").

### The Formula

$$V = \sqrt{\frac{\chi^2}{N \times \min(r-1, c-1)}}$$

Here is what each piece means:

* **$V$**: Cramér's V (the final effect size).
* **$\chi^2$**: The raw Chi-Square statistic you calculated from your contingency table.
* **$N$**: The total number of observations (the Grand Total). Dividing by $N$ is what strips away the sample-size inflation.
* **$r$**: The total number of rows in your table.
* **$c$**: The total number of columns in your table.

> **Note:** If you run a Chi-Square test on an A/B test and find a highly significant p-value, but your Cramér's V is only $0.02$, it tells you a crucial business reality: the feature technically works, but the actual impact it has on user behavior is practically negligible.

In [15]:
def _cramers_v(chi2: float, n: int, shape: tuple) -> float:
    """Cramer's V effect size for a contingency table (0..1)."""
    r, k = shape
    denom = n * (min(r, k) - 1)
    return float(np.sqrt(chi2 / denom)) if denom > 0 else 0.0

In [16]:
max_levels = 20

```python
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)

```

Are the categorical variables in the rows and columns of your table related, or are they completely independent?

* **Step 1: Calculating Expected Frequencies ($E$)** *(a table of what the counts would look like if the Null Hypothesis were perfectly true)*

$$E_{ij} = \frac{\text{Row Total}_i \times \text{Column Total}_j}{\text{Grand Total}}$$


* **Step 2: Computing the Chi-Square Statistic ($\chi^2$)** *(how far reality deviates from the "no effect" baseline)*

$$\chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$


* **Step 3: Calculating Degrees of Freedom ($\text{dof}$)**

$$\text{dof} = (\text{Rows} - 1) \times (\text{Columns} - 1)$$


* **Step 4: Finding the P-Value** *(exact probability of seeing a deviation this large, or larger, purely by random chance if the variables were actually independent)*

```

In [24]:
cat_rows = []
for f in categorical_cols:
    sub = df[[f, click_col]].dropna()
    if sub[f].nunique() < 2 or sub[click_col].nunique() < 2:
        continue

    lvl = sub[f].astype(str)
    if lvl.nunique() > max_levels:                 # bucket rare levels
        top = lvl.value_counts().nlargest(max_levels - 1).index
        lvl = lvl.where(lvl.isin(top), "__OTHER__")

    ct = pd.crosstab(lvl, sub[click_col])

    ctr = sub.assign(_lvl=lvl).groupby("_lvl")[click_col].agg(["mean", "count"])
    best, worst = ctr["mean"].idxmax(), ctr["mean"].idxmin()

    cat_rows.append({
        "feature": f,
        "n_used": len(sub),
        "n_levels": ct.shape[0],
        "cramers_v": _cramers_v(chi2, len(sub), ct.shape),
        "p_chi2": p_chi,
        "best_level": f"{best} (CTR={ctr.loc[best, 'mean']:.3f}, n={int(ctr.loc[best, 'count'])})",
        "worst_level": f"{worst} (CTR={ctr.loc[worst, 'mean']:.3f}, n={int(ctr.loc[worst, 'count'])})",
        # chi-square is unreliable if many expected cell counts are < 5
        "pct_expected_cells_lt5": (expected < 5).mean(),
    })

cat_res = pd.DataFrame(cat_rows)
if not cat_res.empty:
    cat_res["p_adj_BH"] = _benjamini_hochberg(cat_res["p_chi2"])
    cat_res = cat_res.sort_values("cramers_v", ascending=False).reset_index(drop=True)

click        0     1
brand               
acme      6821  1244
globex    5163   722
initech   3674   362
umbrella  1627   387
_____
[[6970.17625 1094.82375]
 [5086.11125  798.88875]
 [3488.113    547.887  ]
 [1740.5995   273.4005 ]]
++++++
click        0     1
device              
desktop   5126   831
mobile   10442  1618
tablet    1717   266
_____
[[ 5148.33725   808.66275]
 [10422.855    1637.145  ]
 [ 1713.80775   269.19225]]
++++++


In [18]:
cat_res

,feature,n_used,n_levels,cramers_v,p_chi2,best_level,worst_level,pct_expected_cells_lt5,p_adj_BH
0,brand,20000,4,0.089351,2.158916e-34,"umbrella (CTR=0.192, n=2014)","initech (CTR=0.090, n=4036)",0.0,4.317832e-34
1,device,20000,3,0.007130,6.014640e-01,"desktop (CTR=0.139, n=5957)","tablet (CTR=0.134, n=1983)",0.0,6.014640e-01
